In [1]:
import math

import torch

##### 1. Define the audio configuration


In [2]:
# The sample rate tells us how many audio samples represent
# one second of audio.
sample_rate = 22_050

In [3]:
# We want to generate one second of artificial audio.
duration_seconds = 1

In [4]:
# n_fft controls the size of the Fourier transform.
# Because the waveform is real-valued, this will produce:
# n_fft // 2 + 1 = 513 unique frequency bins.
n_fft = 1024


In [5]:
# Each STFT frame examines 1,024 waveform samples.
win_length = 1024


In [6]:
# The window moves forward by 256 samples between frames.
hop_length = 256

In [7]:
# When center=True, PyTorch pads both sides of the waveform
# so that each STFT frame is centered at its time position.
center = True


#### 2. Create an artificial one-second waveform


In [8]:
# Number of samples = seconds × samples per second.
#
# 1 second × 22,050 samples/second = 22,050 samples.
waveform_length = duration_seconds * sample_rate

In [9]:
# Fix the random seed so that running the program multiple
# times produces the same artificial waveform.
torch.manual_seed(42)

In [10]:
# torch.rand initially generates values in the range [0, 1).
#
# Multiplying by 2 changes the range to [0, 2).
# Subtracting 1 changes the range to [-1, 1).
#
# The resulting shape is:
# [batch_size, waveform_samples] = [1, 22050]
waveform = torch.rand(
    (1, waveform_length),
    dtype=torch.float32,
) * 2 - 1

In [11]:
# Inspect the waveform before applying the STFT.
print("Waveform shape:", waveform.shape)
print("Waveform dtype:", waveform.dtype)
print("Waveform minimum:", waveform.min().item())
print("Waveform maximum:", waveform.max().item())

Waveform shape: torch.Size([1, 22050])
Waveform dtype: torch.float32
Waveform minimum: -0.9999955892562866
Waveform maximum: 0.9999586343765259


#### 3. Predict the STFT output shape


In [12]:
# For a real-valued waveform, the negative-frequency half
# of the Fourier transform mirrors the positive-frequency
# half. PyTorch therefore keeps only n_fft // 2 + 1 bins.
frequency_bins = n_fft // 2 + 1

In [13]:
# With center=True, PyTorch adds n_fft // 2 samples of
# padding to both the beginning and end of the waveform.
if center:
    padding = n_fft // 2
else:
    padding = 0

In [14]:
# Calculate how many analysis frames will fit.
#
# The numerator represents the effective waveform length
# after padding, minus the size of one analysis frame.
#
# floor is used because a partial hop does not create
# another complete frame.
predicted_frames = 1 + math.floor(
    (waveform_length + 2 * padding - n_fft) / hop_length
)

In [15]:
# The STFT preserves the batch dimension and adds frequency
# and time-frame dimensions:
#
# [B, T_wav] -> [B, frequency_bins, frames]
predicted_shape = (
    waveform.shape[0],
    frequency_bins,
    predicted_frames,
)

In [16]:
print("\nPredicted frequency bins:", frequency_bins)
print("Predicted time frames:", predicted_frames)
print("Predicted STFT shape:", predicted_shape)
 


Predicted frequency bins: 513
Predicted time frames: 87
Predicted STFT shape: (1, 513, 87)


### 4. Create the Hann analysis window

In [17]:
# The STFT divides the waveform into overlapping segments.
#
# Abruptly cutting each segment would create artificial
# frequency components at its boundaries. A Hann window
# smoothly reduces the segment toward zero at its edges.
hann_window = torch.hann_window(
    win_length,

    # The window and waveform must be on the same device.
    device=waveform.device,

    # They should also use the same floating-point dtype.
    dtype=waveform.dtype,
)


### 5. Compute the Short-Time Fourier Transform


In [ ]:
stft_result = torch.stft(
    # Raw audio input with shape [1, 22050].
    waveform,

    # Number of samples used by each Fourier transform.
    n_fft=n_fft,

    # Number of samples between consecutive frames.
    hop_length=hop_length,

    # Number of samples covered by the Hann window.
    win_length=win_length,

    # The Hann window created above.
    window=hann_window,

    # Pad the signal so frames are centered in time.
    center=center,

    # Return complex numbers containing real and imaginary
    # components instead of storing them in another axis.
    return_complex=True,
)

# The expected shape is [1, 513, 87].
print("\nObserved STFT shape:", stft_result.shape)

# A float32 waveform normally produces complex64 STFT values.
print("STFT dtype:", stft_result.dtype)

# Confirm that PyTorch considers this a complex tensor.
print("Is the STFT complex?", torch.is_complex(stft_result))


Observed STFT shape: torch.Size([1, 513, 87])
STFT dtype: torch.complex64
Is the STFT complex? True


In [19]:
# The expected shape is [1, 513, 87].
print("\nObserved STFT shape:", stft_result.shape)

# A float32 waveform normally produces complex64 STFT values.
print("STFT dtype:", stft_result.dtype)

# Confirm that PyTorch considers this a complex tensor.
print("Is the STFT complex?", torch.is_complex(stft_result))


Observed STFT shape: torch.Size([1, 513, 87])
STFT dtype: torch.complex64
Is the STFT complex? True


### 6. Convert the complex STFT into magnitudes


In [20]:
# Every complex STFT value contains real and imaginary parts:
#
# z = real + imaginary × j
#
# Its magnitude is:
#
# magnitude = sqrt(real² + imaginary²)
#
# Calling abs() calculates this magnitude and discards phase
# information.
magnitude = stft_result.abs()

# Taking the magnitude changes the values from complex64 to
# float32, but it does not change the tensor's shape.
print("\nMagnitude shape:", magnitude.shape)
print("Magnitude dtype:", magnitude.dtype)

# Magnitudes represent strength and therefore cannot be
# negative.
all_nonnegative = torch.all(magnitude >= 0)
print("All magnitudes are nonnegative:", bool(all_nonnegative))


Magnitude shape: torch.Size([1, 513, 87])
Magnitude dtype: torch.float32
All magnitudes are nonnegative: True


In [21]:
# 7. Verify all expected properties
# ---------------------------------------------------------

# Check that our manual shape prediction was correct.
assert stft_result.shape == predicted_shape

# Taking the magnitude must not change the shape.
assert magnitude.shape == stft_result.shape

# The STFT should contain complex values.
assert torch.is_complex(stft_result)

# The magnitude should contain ordinary real values.
assert not torch.is_complex(magnitude)

# Every magnitude value must be zero or positive.
assert torch.all(magnitude >= 0)

print("\nAll checks passed.")


All checks passed.
